In [56]:
# ==========================================================
# Built-in Python Libraries
# ==========================================================
import os
import re
import string

# ==========================================================
# Data Handling
# ==========================================================
import pandas as pd

# ==========================================================
# Visualization
# ==========================================================
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================================
# HTML Processing
# ==========================================================
from bs4 import BeautifulSoup

# ==========================================================
# Text Processing
# ==========================================================
import contractions
import emoji

# ==========================================================
# Spell Correction (SymSpell)
# ==========================================================
from symspellpy import SymSpell
import pkg_resources

# ==========================================================
# NLTK
# ==========================================================
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.util import ngrams

# ==========================================================
# String Similarity
# ==========================================================
from difflib import SequenceMatcher

# ==========================================================
# Machine Learning (Scikit-learn)
# ==========================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================================
# Transformer Model (SBERT)
# ==========================================================
from sentence_transformers import SentenceTransformer

In [57]:
# ==========================================================
# Download NLTK Resources
# ==========================================================
resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger_eng"
]

for resource in resources:
    nltk.download(resource)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\Think\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       d

In [58]:
# ==========================================================
# Load SBERT Model
# ==========================================================
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [59]:
def process_data(text):
       
    text = BeautifulSoup(text, "html.parser").text
          
    text = contractions.fix(text)
     
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    text = re.sub(r'\S+@\S+', '', text)
    
    text = re.sub(r'\+?\d[\d\s-]{8,}\d', '', text)
       
    text = emoji.replace_emoji(text, replace='')
    
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    text = text.lower()
    
    dictionary_path = pkg_resources.resource_filename(
        "symspellpy",
        "frequency_dictionary_en_82_765.txt"
    )
    
    # Create object
    sym_spell = SymSpell(max_dictionary_edit_distance=2)
    
    # Load dictionary
    dictionary_path = pkg_resources.resource_filename(
        "symspellpy",
        "frequency_dictionary_en_82_765.txt"
    )
    
    sym_spell.load_dictionary(
        dictionary_path,
        term_index=0,
        count_index=1
    )
    
    
    suggestions = sym_spell.lookup_compound(
        text,
        max_edit_distance=2
    )
    
    text=suggestions[0].term
    
    tokens = word_tokenize(text)
    
    stop_words = set(stopwords.words('english'))
    
    tokens = [word for word in tokens if word not in stop_words]
    
    lemmatizer = WordNetLemmatizer()
    
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    text = " ".join(tokens)
    
    return text

In [60]:
def ngram_similarity(text1, text2, n=2):

    grams1 = set(ngrams(text1.split(), n))
    grams2 = set(ngrams(text2.split(), n))

    intersection = grams1.intersection(grams2)
    union = grams1.union(grams2)

    if len(union) == 0:
        return 0

    return len(intersection) / len(union)

In [61]:
def tfidf_similarity(text1, text2):
    
    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform([text1, text2])

    score = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])[0][0]

    return score

In [62]:
def sequence_similarity(text1, text2):

    score = SequenceMatcher(None, text1, text2).ratio()

    return score

In [63]:
def sbert_similarity(text1, text2):

    embeddings = model.encode([text1, text2])

    score = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )[0][0]

    return max(0, score)

In [64]:
def hybrid_similarity(text1, text2):

    tfidf = tfidf_similarity(text1, text2)
    ngram = ngram_similarity(text1, text2, n=2)
    sequence = sequence_similarity(text1, text2)
    sbert = max(0, sbert_similarity(text1, text2))

    final = (
          0.25 * tfidf +
          0.20 * ngram +
          0.20 * sequence +
          0.35 * sbert
    )

    if final >= 0.90:
        verdict = "Exact Copy"
    elif final >= 0.70:
        verdict = "Highly Similar"
    elif final >= 0.50:
        verdict = "Possibly Plagiarized"
    elif final >= 0.30:
        verdict = "Moderately Similar"
    else:
        verdict = "Different"

    return {
        "TF-IDF": tfidf,
        "N-Gram": ngram,
        "Sequence": sequence,
        "SBERT": sbert,
        "Final": final,
        "Verdict": verdict
    }

In [65]:
folder_path = r"C:\Users\Think\Desktop\anacondaaa\nlp\NLP Models\plagiarism"

documents = {}

for file in os.listdir(folder_path):
    if file.endswith(".txt"):
        file_path = os.path.join(folder_path, file)

        with open(file_path, "r", encoding="utf-8") as f:
            documents[file] = f.read()

print("Total Documents:", len(documents))

Total Documents: 2


In [66]:
processed_documents = {}
for filename, text in documents.items():
    processed_documents[filename] = process_data(text)

filename = list(documents.keys())[0]



In [67]:
print(len(processed_documents))

2


In [68]:
filenames = list(processed_documents.keys())

doc1 = processed_documents[filenames[0]]
doc2 = processed_documents[filenames[1]]

result = hybrid_similarity(doc1, doc2)

In [69]:
print(f"TF-IDF            : {result['TF-IDF']*100:.2f}%")
print(f"N-Gram            : {result['N-Gram']*100:.2f}%")
print(f"SequenceMatcher   : {result['Sequence']*100:.2f}%")
print(f"SBERT             : {result['SBERT']*100:.2f}%")
print("-" * 40)
print(f"Final Score       : {result['Final']*100:.2f}%")
print(f"Verdict           : {result['Verdict']}")

TF-IDF            : 74.10%
N-Gram            : 11.56%
SequenceMatcher   : 3.16%
SBERT             : 93.47%
----------------------------------------
Final Score       : 54.18%
Verdict           : Possibly Plagiarized
